# Fit MIMIC image embeddings

Load a serialized vision dataset, fit MIMIC on the flattened image rows, compute embeddings with `transform`, and save those embeddings for later visualization.

In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "vision" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ROOT_SRC = PROJECT_ROOT / "src"
VISION_SRC = PROJECT_ROOT / "vision" / "src"
for src_dir in [str(ROOT_SRC), str(VISION_SRC)]:
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

from mimic import MIMIC
from mimic_vision import (
    VisionEmbedding,
    load_serialized_vision_dataset,
    resolve_artifact_file,
    save_vision_embedding,
)

In [35]:
DATASET_FILE = "last"  # Use "last" or an explicit dataset filename.
DATASET_DIR = PROJECT_ROOT / "vision" / "data" / "serialized"
EMBEDDING_DIR = PROJECT_ROOT / "vision" / "data" / "embeddings"
MODEL_DIR = PROJECT_ROOT / "vision" / "data" / "models"

MODE = "direct"  # options: "identity", "direct", "factorised", "joint"
CAPACITY = 0.2
RANDOM_STATE = 0
BOOTSTRAP = False
FEATURE_N_JOBS = -1

In [36]:
dataset_path = resolve_artifact_file(DATASET_FILE, input_dir=DATASET_DIR, pattern="*.pkl")
DATASET_FILE = dataset_path.name
dataset = load_serialized_vision_dataset(dataset_path)
DATASET_FILE, dataset.X.shape, dataset.images.shape, dataset.y.value_counts().sort_index()

('mnist_train_n3200_classes-5-6-8-9_21x21.pkl',
 (3200, 441),
 (3200, 21, 21),
 target
 5    800
 6    800
 8    800
 9    800
 Name: count, dtype: int64)

In [37]:
columns = {
    "regression": list(dataset.X.columns),
    "classification": [],
    "ignore": [],
}

model = MIMIC(
    columns=columns,
    mode=MODE,
    capacity=CAPACITY,
    bootstrap=BOOTSTRAP,
    feature_n_jobs=FEATURE_N_JOBS,
    random_state=RANDOM_STATE,
)
model.fit(dataset.X)

MIMIC(bootstrap=False, capacity=0.2,
      columns={'classification': [], 'ignore': [],
               'regression': ['px_0000', 'px_0001', 'px_0002', 'px_0003',
                              'px_0004', 'px_0005', 'px_0006', 'px_0007',
                              'px_0008', 'px_0009', 'px_0010', 'px_0011',
                              'px_0012', 'px_0013', 'px_0014', 'px_0015',
                              'px_0016', 'px_0017', 'px_0018', 'px_0019',
                              'px_0020', 'px_0021', 'px_0022', 'px_0023',
                              'px_0024', 'px_0025', 'px_0026', 'px_0027',
                              'px_0028', 'px_0029', ...]},
      feature_n_jobs=-1, mode='direct', random_state=0)

In [38]:
embeddings = model.transform(dataset.X)
embeddings.shape

(3200, 11466)

In [39]:
embedding_artifact = VisionEmbedding(
    dataset_file=DATASET_FILE,
    embeddings=embeddings,
    mode=MODE,
    capacity=CAPACITY,
    random_state=RANDOM_STATE,
)
saved_embedding_path = save_vision_embedding(embedding_artifact, output_dir=EMBEDDING_DIR)
saved_embedding_path.name

'mnist_train_n3200_classes-5-6-8-9_21x21_mimic-direct_cap0p2_emb11466.pkl'

In [40]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_filename = saved_embedding_path.with_suffix(".joblib").name
saved_model_path = MODEL_DIR / model_filename
model.save(saved_model_path)
saved_model_path.name

'mnist_train_n3200_classes-5-6-8-9_21x21_mimic-direct_cap0p2_emb11466.joblib'